In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Batch Model Serving
# MAGIC Loads both @champion models from Unity Catalog and scores current data,
# MAGIC writing predictions to Gold tables.

In [0]:
%pip install lightgbm xgboost
dbutils.library.restartPython()

In [0]:
import mlflow
import pandas as pd

catalog = "credit_risk_fraud_detection"
mlflow.set_registry_uri("databricks-uc")

In [0]:
# MAGIC %md ## 1. Fraud model — score all transactions

In [0]:
fraud_model = mlflow.xgboost.load_model(f"models:/{catalog}.ml.fraud_classifier_model@champion")

max_date = spark.sql(f"""
    select max(transaction_date) as max_date from {catalog}.silver.fct_transactions
""").collect()[0]["max_date"]
print("Most recent transaction date in dataset:", max_date)

txns_spark = spark.sql(f"""
    select * from {catalog}.silver.fct_transactions
    where transaction_date >= date_sub('{max_date}', 30)
""")
print(txns_spark.count(), "transactions in scoring window")

txns = txns_spark.toPandas()

categorical_cols = ["transaction_type", "transaction_mode", "channel"]
numeric_cols = [
    "amount", "distance_from_home_km",
    "velocity_1hr", "velocity_24hr", "amount_vs_30d_avg_ratio", "time_since_last_txn_mins",
]
bool_cols = ["is_new_device", "is_new_merchant_category", "is_night_transaction"]

for c in numeric_cols:
    txns[c] = pd.to_numeric(txns[c], errors="coerce")
for c in bool_cols:
    txns[c] = txns[c].astype(bool)
for c in categorical_cols:
    txns[c] = txns[c].astype("category")   # plain string, matches the logged model signature -- NOT category dtype

feature_cols = numeric_cols + bool_cols + categorical_cols
X = txns[feature_cols]

txns["fraud_probability"] = fraud_model.predict_proba(X)[:, 1]
txns["fraud_flag_predicted"] = txns["fraud_probability"] >= 0.5

fraud_predictions = txns[[
    "transaction_id", "customer_id", "transaction_date", "amount",
    "is_fraud", "fraud_probability", "fraud_flag_predicted",
]]

spark.createDataFrame(fraud_predictions).write.mode("overwrite").saveAsTable(
    f"{catalog}.gold.fraud_predictions"
)
print(spark.table(f"{catalog}.gold.fraud_predictions").count(), "transactions scored")

In [0]:
# MAGIC %md ## 2. Delinquency model — score currently-active loans (latest month, no label needed)

In [0]:
delinquency_model = mlflow.lightgbm.load_model(f"models:/{catalog}.ml.delinquency_escalation_model@champion")

scoring_df = spark.sql(f"""
WITH loan_static AS (
    select distinct
        loan_account_id, customer_id, loan_type, interest_rate, tenure_months,
        emi_amount, disbursed_amount, cibil_score_at_decision
    from {catalog}.silver.dim_loan_account
),
persona_lookup as (
    select customer_id, persona from {catalog}.silver.customer_snapshot where dbt_valid_to is null
),
events as (
    select loan_account_id, customer_id, installment_number, due_date,
           dpd_at_event, loan_status_at_event, consecutive_missed_at_event,
           is_missed, is_partial, salary_credited_this_month
    from {catalog}.silver.fct_repayment_events
),
rolling as (
    select *,
        sum(case when is_missed then 1 else 0 end) over (
            partition by loan_account_id order by installment_number
            rows between 3 preceding and 1 preceding) as missed_last_3m,
        sum(case when is_missed then 1 else 0 end) over (
            partition by loan_account_id order by installment_number
            rows between 6 preceding and 1 preceding) as missed_last_6m,
        sum(case when is_partial then 1 else 0 end) over (
            partition by loan_account_id order by installment_number
            rows between 3 preceding and 1 preceding) as partial_last_3m,
        row_number() over (
            partition by loan_account_id order by installment_number desc) as rn
    from events
),
bounce_counts as (
    select loan_account_id, nach_debit_date, consecutive_bounce_count
    from {catalog}.silver.fct_nach_bounces
),
latest_with_bounces as (
    select r.*,
        (select max(b.consecutive_bounce_count) from bounce_counts b
         where b.loan_account_id = r.loan_account_id and b.nach_debit_date <= r.due_date
        ) as bounce_count_as_of_month
    from rolling r
    where r.rn = 1
      and r.loan_status_at_event in ('ACTIVE', 'CURRENT', '30_DPD')
)

select
    l.loan_account_id, l.customer_id, ls.loan_type, ls.interest_rate, ls.tenure_months,
    ls.emi_amount, ls.disbursed_amount, ls.cibil_score_at_decision, p.persona,
    l.installment_number, l.due_date, l.dpd_at_event, l.loan_status_at_event,
    l.consecutive_missed_at_event, l.salary_credited_this_month,
    coalesce(l.missed_last_3m, 0) as missed_last_3m,
    coalesce(l.missed_last_6m, 0) as missed_last_6m,
    coalesce(l.partial_last_3m, 0) as partial_last_3m,
    coalesce(l.bounce_count_as_of_month, 0) as bounce_count_as_of_month
from latest_with_bounces l
join loan_static ls on l.loan_account_id = ls.loan_account_id
left join persona_lookup p on l.customer_id = p.customer_id
""").toPandas()

print(len(scoring_df), "currently-active loans to score")

for c in ["loan_type", "persona", "loan_status_at_event"]:
    scoring_df[c] = scoring_df[c].astype("category")

feature_cols_delinq = [
    "interest_rate", "tenure_months", "emi_amount", "disbursed_amount",
    "cibil_score_at_decision", "dpd_at_event", "consecutive_missed_at_event",
    "missed_last_3m", "missed_last_6m", "partial_last_3m",
    "bounce_count_as_of_month", "salary_credited_this_month",
    "loan_type", "persona", "loan_status_at_event",
]

scoring_df["escalation_probability"] = delinquency_model.predict_proba(scoring_df[feature_cols_delinq])[:, 1]

delinquency_predictions = scoring_df[[
    "loan_account_id", "customer_id", "loan_type", "persona",
    "due_date", "dpd_at_event", "loan_status_at_event", "escalation_probability",
]]

spark.createDataFrame(delinquency_predictions).write.mode("overwrite").saveAsTable(
    f"{catalog}.gold.delinquency_risk_scores"
)
print(spark.table(f"{catalog}.gold.delinquency_risk_scores").count(), "loans scored")

In [0]:
display(spark.sql(f"""
    select loan_status_at_event, round(avg(escalation_probability), 4) as avg_risk_score, count(*) as n
    from {catalog}.gold.delinquency_risk_scores
    group by loan_status_at_event
    order by avg_risk_score desc
"""))